# CLIP From Scratch in PyTorch

This notebook implements a compact CLIP-style model on FashionMNIST, following the educational structure of Matt Nguyen’s Medium article **“Building CLIP From Scratch”**, but with independently written code and practical corrections.

Article: https://medium.com/correll-lab/building-clip-from-scratch-68f6e42d35f4

It includes a byte tokenizer, ViT image encoder, Transformer text encoder, shared embedding space, symmetric contrastive loss, training, evaluation, retrieval, and zero-shot-style classification.

## 1. Imports

In [ ]:
import math, random
from dataclasses import dataclass
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.datasets import FashionMNIST

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))

## 2. Configuration

In [ ]:
@dataclass
class CLIPConfig:
    image_size:int=28
    patch_size:int=7
    image_channels:int=1
    image_width:int=96
    image_layers:int=4
    image_heads:int=4
    vocab_size:int=256
    max_text_length:int=32
    text_width:int=96
    text_layers:int=4
    text_heads:int=4
    projection_dim:int=64
    mlp_ratio:int=4
    dropout:float=0.1
    batch_size:int=128
    epochs:int=10
    learning_rate:float=3e-4
    weight_decay:float=1e-4
    num_workers:int=2

config=CLIPConfig()
assert config.image_size % config.patch_size == 0
assert config.image_width % config.image_heads == 0
assert config.text_width % config.text_heads == 0
print("Patches:", (config.image_size//config.patch_size)**2)

## 3. Captions and byte tokenizer

Token IDs 0, 2 and 3 represent padding, start-of-text and end-of-text. A production CLIP model would normally use BPE or another subword tokenizer.

In [ ]:
CLASS_NAMES=["t-shirt/top","trousers","pullover","dress","coat","sandal","shirt","sneaker","bag","ankle boot"]
CAPTION_TEMPLATES=["an image of a {}","a photo of a {}","a grayscale image of a {}"]

class ByteTokenizer:
    PAD_ID=0; SOT_ID=2; EOT_ID=3
    def __init__(self,max_length=32): self.max_length=max_length
    def encode(self,text):
        body=list(text.lower().encode("utf-8"))[:self.max_length-2]
        ids=[self.SOT_ID]+body+[self.EOT_ID]
        mask=[1]*len(ids)
        pad=self.max_length-len(ids)
        ids += [self.PAD_ID]*pad; mask += [0]*pad
        return torch.tensor(ids,dtype=torch.long), torch.tensor(mask,dtype=torch.bool)
    def batch_encode(self,texts):
        items=[self.encode(t) for t in texts]
        return torch.stack([x[0] for x in items]), torch.stack([x[1] for x in items])

tokenizer=ByteTokenizer(config.max_text_length)
print(tokenizer.encode("an image of a sneaker")[0].shape)

## 4. FashionMNIST image–caption dataset

In [ ]:
class FashionMNISTCLIPDataset(Dataset):
    def __init__(self,root,train,tokenizer,random_templates=True):
        self.dataset=FashionMNIST(root=root,train=train,download=True,transform=transforms.Compose([transforms.ToTensor(),transforms.Normalize((0.2860,),(0.3530,))]))
        self.tokenizer=tokenizer; self.random_templates=random_templates
    def __len__(self): return len(self.dataset)
    def __getitem__(self,index):
        image,label=self.dataset[index]
        template=random.choice(CAPTION_TEMPLATES) if self.random_templates else CAPTION_TEMPLATES[0]
        token_ids,attention_mask=self.tokenizer.encode(template.format(CLASS_NAMES[label]))
        return {"image":image,"text":token_ids,"attention_mask":attention_mask,"label":torch.tensor(label)}

train_dataset=FashionMNISTCLIPDataset("./data",True,tokenizer,True)
test_dataset=FashionMNISTCLIPDataset("./data",False,tokenizer,False)
kwargs=dict(batch_size=config.batch_size,num_workers=config.num_workers,pin_memory=torch.cuda.is_available())
train_loader=DataLoader(train_dataset,shuffle=True,drop_last=True,**kwargs)
test_loader=DataLoader(test_dataset,shuffle=False,drop_last=False,**kwargs)
batch=next(iter(train_loader))
for k,v in batch.items(): print(k,tuple(v.shape))

## 5. Sinusoidal positional encoding

In [ ]:
def build_sinusoidal_encoding(length,width):
    pos=torch.arange(length,dtype=torch.float32).unsqueeze(1)
    freq=torch.exp(torch.arange(0,width,2,dtype=torch.float32)*(-math.log(10000.0)/width))
    pe=torch.zeros(length,width)
    pe[:,0::2]=torch.sin(pos*freq)
    pe[:,1::2]=torch.cos(pos*freq[:pe[:,1::2].shape[1]])
    return pe.unsqueeze(0)

class SinusoidalPositionEncoding(nn.Module):
    def __init__(self,length,width):
        super().__init__(); self.register_buffer("encoding",build_sinusoidal_encoding(length,width),persistent=False)
    def forward(self,x): return x+self.encoding[:,:x.shape[1]]

## 6. Shared Transformer encoder block

In [ ]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self,width,num_heads,mlp_ratio=4,dropout=0.0):
        super().__init__(); hidden=width*mlp_ratio
        self.norm1=nn.LayerNorm(width)
        self.attention=nn.MultiheadAttention(width,num_heads,dropout=dropout,batch_first=True)
        self.norm2=nn.LayerNorm(width)
        self.mlp=nn.Sequential(nn.Linear(width,hidden),nn.GELU(),nn.Dropout(dropout),nn.Linear(hidden,width),nn.Dropout(dropout))
    def forward(self,x,padding_mask=None):
        z=self.norm1(x)
        attn,_=self.attention(z,z,z,key_padding_mask=padding_mask,need_weights=False)
        x=x+attn
        return x+self.mlp(self.norm2(x))

## 7. Vision Transformer image encoder

In [ ]:
class ImageEncoder(nn.Module):
    def __init__(self,cfg):
        super().__init__(); self.num_patches=(cfg.image_size//cfg.patch_size)**2
        self.patch_projection=nn.Conv2d(cfg.image_channels,cfg.image_width,kernel_size=cfg.patch_size,stride=cfg.patch_size)
        self.class_token=nn.Parameter(torch.zeros(1,1,cfg.image_width))
        self.position=SinusoidalPositionEncoding(self.num_patches+1,cfg.image_width)
        self.dropout=nn.Dropout(cfg.dropout)
        self.blocks=nn.ModuleList([TransformerEncoderBlock(cfg.image_width,cfg.image_heads,cfg.mlp_ratio,cfg.dropout) for _ in range(cfg.image_layers)])
        self.norm=nn.LayerNorm(cfg.image_width)
        self.projection=nn.Linear(cfg.image_width,cfg.projection_dim,bias=False)
        nn.init.normal_(self.class_token,std=0.02)
    def forward(self,images):
        x=self.patch_projection(images).flatten(2).transpose(1,2)
        cls=self.class_token.expand(images.shape[0],-1,-1)
        x=self.dropout(self.position(torch.cat([cls,x],dim=1)))
        for block in self.blocks: x=block(x)
        return F.normalize(self.projection(self.norm(x[:,0])),dim=-1)

## 8. Transformer text encoder

In [ ]:
class TextEncoder(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.token_embedding=nn.Embedding(cfg.vocab_size,cfg.text_width,padding_idx=ByteTokenizer.PAD_ID)
        self.position=SinusoidalPositionEncoding(cfg.max_text_length,cfg.text_width)
        self.dropout=nn.Dropout(cfg.dropout)
        self.blocks=nn.ModuleList([TransformerEncoderBlock(cfg.text_width,cfg.text_heads,cfg.mlp_ratio,cfg.dropout) for _ in range(cfg.text_layers)])
        self.norm=nn.LayerNorm(cfg.text_width)
        self.projection=nn.Linear(cfg.text_width,cfg.projection_dim,bias=False)
    def forward(self,token_ids,attention_mask):
        x=self.dropout(self.position(self.token_embedding(token_ids)))
        padding_mask=~attention_mask.bool()
        for block in self.blocks: x=block(x,padding_mask)
        x=self.norm(x)
        end_positions=attention_mask.long().sum(dim=1)-1
        rows=torch.arange(token_ids.shape[0],device=token_ids.device)
        return F.normalize(self.projection(x[rows,end_positions]),dim=-1)

## 9. CLIP model and symmetric contrastive loss

In [ ]:
class CLIPModel(nn.Module):
    def __init__(self,cfg):
        super().__init__(); self.image_encoder=ImageEncoder(cfg); self.text_encoder=TextEncoder(cfg)
        self.logit_scale=nn.Parameter(torch.tensor(math.log(1/0.07)))
    def encode_image(self,images): return self.image_encoder(images)
    def encode_text(self,ids,mask): return self.text_encoder(ids,mask)
    def forward(self,images,ids,mask):
        i=self.encode_image(images); t=self.encode_text(ids,mask)
        scale=self.logit_scale.exp().clamp(max=100)
        logits=scale*i@t.T
        return logits,logits.T

def clip_loss(logits_i,logits_t):
    targets=torch.arange(logits_i.shape[0],device=logits_i.device)
    return 0.5*(F.cross_entropy(logits_i,targets)+F.cross_entropy(logits_t,targets))

model=CLIPModel(config).to(device)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
with torch.no_grad():
    li,lt=model(batch["image"][:8].to(device),batch["text"][:8].to(device),batch["attention_mask"][:8].to(device))
print(li.shape,lt.shape)

## 10. Training utilities

In [ ]:
def train_one_epoch(model,loader,optimizer,device):
    model.train(); total_loss=0.0; total=0
    for b in loader:
        images=b["image"].to(device,non_blocking=True); ids=b["text"].to(device,non_blocking=True); mask=b["attention_mask"].to(device,non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        li,lt=model(images,ids,mask); loss=clip_loss(li,lt)
        loss.backward(); optimizer.step()
        total_loss += loss.item()*images.shape[0]; total += images.shape[0]
    return total_loss/total

@torch.inference_mode()
def evaluate_loss(model,loader,device):
    model.eval(); total_loss=0.0; total=0
    for b in loader:
        images=b["image"].to(device); ids=b["text"].to(device); mask=b["attention_mask"].to(device)
        li,lt=model(images,ids,mask); loss=clip_loss(li,lt)
        total_loss += loss.item()*images.shape[0]; total += images.shape[0]
    return total_loss/total

## 11. Train CLIP

In [ ]:
optimizer=AdamW(model.parameters(),lr=config.learning_rate,weight_decay=config.weight_decay)
history={"train":[],"test":[]}
best=float("inf"); checkpoint_path="clip_fashionmnist_from_scratch.pt"
for epoch in range(config.epochs):
    train_loss=train_one_epoch(model,train_loader,optimizer,device)
    test_loss=evaluate_loss(model,test_loader,device)
    history["train"].append(train_loss); history["test"].append(test_loss)
    print(f"Epoch {epoch+1:02d}/{config.epochs} | train {train_loss:.4f} | test {test_loss:.4f} | scale {model.logit_scale.exp().item():.2f}")
    if test_loss<best:
        best=test_loss
        torch.save({"model_state_dict":model.state_dict(),"config":vars(config),"best_test_loss":best},checkpoint_path)
        print("Saved best model")

## 12. Plot training loss

In [ ]:
x=range(1,len(history["train"])+1)
plt.figure(figsize=(8,5)); plt.plot(x,history["train"],marker="o",label="Train"); plt.plot(x,history["test"],marker="o",label="Test")
plt.xlabel("Epoch"); plt.ylabel("Contrastive loss"); plt.title("CLIP training history"); plt.legend(); plt.show()

## 13. Build text embeddings for all classes

In [ ]:
@torch.inference_mode()
def build_class_embeddings(model,class_names,templates,tokenizer,device):
    model.eval(); outputs=[]
    for name in class_names:
        prompts=[t.format(name) for t in templates]
        ids,mask=tokenizer.batch_encode(prompts)
        emb=model.encode_text(ids.to(device),mask.to(device)).mean(dim=0)
        outputs.append(F.normalize(emb,dim=0))
    return torch.stack(outputs)

checkpoint=torch.load(checkpoint_path,map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
class_text_embeddings=build_class_embeddings(model,CLASS_NAMES,CAPTION_TEMPLATES,tokenizer,device)
print(class_text_embeddings.shape)

## 14. Zero-shot-style classification

In [ ]:
@torch.inference_mode()
def zero_shot_accuracy(model,loader,class_embeddings,device):
    model.eval(); correct=total=0
    for b in loader:
        images=b["image"].to(device); labels=b["label"].to(device)
        pred=(model.encode_image(images)@class_embeddings.T).argmax(dim=1)
        correct += (pred==labels).sum().item(); total += labels.numel()
    return correct/total

accuracy=zero_shot_accuracy(model,test_loader,class_text_embeddings,device)
print(f"Zero-shot-style accuracy: {accuracy*100:.2f}%")

## 15. Visualize predictions

In [ ]:
@torch.inference_mode()
def show_predictions(model,dataset,class_embeddings,indices,device):
    images=torch.stack([dataset[i]["image"] for i in indices]).to(device)
    labels=torch.stack([dataset[i]["label"] for i in indices])
    probs=(model.encode_image(images)@class_embeddings.T).softmax(dim=-1)
    preds=probs.argmax(dim=1).cpu()
    fig,axes=plt.subplots(1,len(indices),figsize=(3*len(indices),3))
    if len(indices)==1: axes=[axes]
    for j,ax in enumerate(axes):
        ax.imshow(images[j].cpu().squeeze(0),cmap="gray")
        p=preds[j].item(); ax.set_title(f"True: {CLASS_NAMES[labels[j]]}\nPred: {CLASS_NAMES[p]}\nScore: {probs[j,p].item():.3f}"); ax.axis("off")
    plt.tight_layout(); plt.show()

show_predictions(model,test_dataset,class_text_embeddings,[0,1,2,3,4],device)

## Architecture summary

```text
Image -> Conv2D patches -> class token + position -> ViT -> projection -> normalized image embedding
Text -> byte tokens -> token embedding + position -> Transformer -> EOT token -> projection -> normalized text embedding

image embeddings @ text embeddings.T -> pairwise similarities -> symmetric contrastive loss
```